# Cox Proportional Hazards Survival Models

Converted from `Code/r_03_survival_analysis.R` — R-kernel Jupyter port. Outputs tables to `Output/Tables/` and figures to `Output/Images/Graphs/`. Run the setup cell first, then sections in order.

In [1]:
pacman::p_load(
  sf, tidyverse, stargazer, spatialreg, spatstat, sp,
  raster, spdep, conleyreg, dplyr, survival, survminer,
  ggplot2, broom, jsonlite
)

PROJECT_ROOT <- tryCatch(
  normalizePath(file.path(dirname(rstudioapi::getActiveDocumentContext()$path), "..")),
  error = function(e) {
    cwd <- normalizePath(getwd())
    if (basename(cwd) == "Code") dirname(cwd) else cwd
  }
)
setwd(PROJECT_ROOT)
# Load pretty dictionary for labels
pretty_dict <- fromJSON("Code/pretty_dict.json")

pdf <- read_sf(dsn = "Data/Processed/northParishFlows.shp")

day <- 40
rdf <- data.frame(pdf)
rdf$day <- replace(rdf$day, rdf$day < 1, day)
rdf$day <- ifelse(is.na(rdf$day), day, rdf$day)
rdf$primary <- ifelse(is.na(rdf$primary), 0, rdf$primary)
rdf$primary_day <- rdf$day * rdf$primary
rdf$primary_day <- replace(rdf$primary_day, rdf$primary_day < 1, day)

rdf$survival <- rdf$day - rdf$news_day
rdf$primary_survival <- rdf$primary_day - rdf$news_day
rdf$primary_survival <- ifelse(is.na(rdf$primary_survival), day, rdf$primary_survival)

# Standardize and center continuous variables.
# Binary dummies (smHouse, bigHouse, mg_fsnub, mg_court, friary) are NOT standardized.
for (v in c(
  # Total monastic land (3 normalizations)
  "llandOwned", "llo_sk", "llo_arak",
  # Small/large split land (3 normalizations)
  "lsmLand",   "lbigLand",
  "lsm_sk",    "lbg_sk",
  "lsm_arak",  "lbg_arak",
  # Off-site/on-site split land (3 normalizations)
  "lotherLand", "lownLand",
  "loth_sk",    "lown_sk",
  "loth_arak",  "lown_arak",
  # Tithes, alms, net income (3 normalizations each)
  "ltitheOutT", "lti_sk", "lti_arak",
  "lalmsInTot", "lal_sk", "lal_arak",
                "lni_sk", "lni_arak",
  # Controls (continuous)
  "lLStax_pc", "lpopC", "distScot", "area", "mean_slope",
  "wet_1535", "wet_1536"
)) {
  rdf[[v]] <- scale(rdf[[v]], center = TRUE, scale = TRUE)[, 1]
}

# Nine monastic-variable specifications: 3 structures × 3 normalizations
cox_mon_specs <- list(
  total_raw  = c("llandOwned",              "ltitheOutT", "lalmsInTot"),
  total_sk   = c("llo_sk",                  "lti_sk",     "lal_sk"),
  total_arak = c("llo_arak",                "lti_arak",   "lal_arak",  "lni_arak"),
  split_raw  = c("lsmLand",  "lbigLand",    "ltitheOutT", "lalmsInTot"),
  split_sk   = c("lsm_sk",   "lbg_sk",      "lti_sk",     "lal_sk"),
  split_arak = c("lsm_arak", "lbg_arak",    "lti_arak",   "lal_arak",  "lni_arak"),
  ownOther_raw  = c("lotherLand", "lownLand",  "ltitheOutT", "lalmsInTot"),
  ownOther_sk   = c("loth_sk",    "lown_sk",   "lti_sk",     "lal_sk"),
  ownOther_arak = c("loth_arak",  "lown_arak", "lti_arak",   "lal_arak", "lni_arak")
)

cox_spec_suffix <- list(
  total_raw     = "_total_raw",    total_sk  = "_total_sk",   total_arak  = "_total_arak",
  split_raw     = "_split_raw",    split_sk  = "_split_sk",   split_arak  = "_split_arak",
  ownOther_raw  = "_ownOther_raw", ownOther_sk = "_ownOther_sk",
  ownOther_arak = "_ownOther_arak"
)


In [2]:
# Build a spatial version of rdf so conleyreg can compute inter-parish distances.
# rdf is data.frame(pdf) with survival columns appended; we re-attach geometry.
rdf_sf <- sf::st_sf(rdf, geometry = sf::st_geometry(pdf))

# Helper: Conley (spatial HAC) SEs at 100 km for a coxph model.
# conleyreg supports model = "cox" with a Surv() LHS formula.
conley_cox_se <- function(cox_mod, cutoff_km = 100) {
  fm <- formula(cox_mod)
  cr <- tryCatch(
    conleyreg(
      formula     = fm,
      data        = rdf_sf,
      dist_cutoff = cutoff_km,
      model       = "cox",
      kernel      = "bartlett",
      verbose     = FALSE
    ),
    error = function(e) {
      cat("conleyreg (cox) failed:", conditionMessage(e), "\n"); NULL
    }
  )
  if (is.null(cr)) return(NULL)
  out <- setNames(rep(NA_real_, length(coef(cox_mod))), names(coef(cox_mod)))
  out[rownames(cr)] <- cr[, "Std. Error"]
  out
}

## Section 1: Cox Proportional Hazards Models (stargazer tables)

In [3]:
# Fit cox1 / cox2 / cox3 for each monastic-variable specification.
# cox_middle: standard monastic controls (binary proximity dummies + friary)
# cox_elite:  elite proximity controls (20km binary dummies)
# Progressive models:
#   m1: monastic block + weather
#   m2: + elite + taxation/population
#   m3: + geographic controls
# `cox1, cox2, cox3` aliases for the split_arak spec are exposed below so that
# downstream cells that reference them by name keep working.

cox_middle  <- c("smHouse", "bigHouse", "friary")
cox_elite   <- c("mg_fsnub", "mg_court")
cox_weather <- c("wet_1535", "wet_1536")
cox_taxpop  <- c("lLStax_pc", "lpopC")
cox_geo     <- c("distScot", "area", "uplands", "lowlands", "mean_slope")

fit_cox_trio <- function(mon_vars) {
  rhs1 <- paste(c(mon_vars, cox_middle, cox_weather), collapse = " + ")
  rhs2 <- paste(c(mon_vars, cox_middle, cox_elite,
                  cox_weather, cox_taxpop), collapse = " + ")
  rhs3 <- paste(c(mon_vars, cox_middle, cox_elite,
                  cox_weather, cox_taxpop, cox_geo), collapse = " + ")
  list(
    m1 = coxph(as.formula(paste("Surv(primary_survival, primary) ~", rhs1)), data = rdf),
    m2 = coxph(as.formula(paste("Surv(primary_survival, primary) ~", rhs2)), data = rdf),
    m3 = coxph(as.formula(paste("Surv(primary_survival, primary) ~", rhs3)), data = rdf)
  )
}

cox_fits <- lapply(cox_mon_specs, fit_cox_trio)

# Back-compat aliases: split_arak is the primary per-arable-km² specification
cox1 <- cox_fits$split_arak$m1
cox2 <- cox_fits$split_arak$m2
cox3 <- cox_fits$split_arak$m3


## Proportional Hazards Assumption — Schoenfeld Residual Tests

Null hypothesis: log hazard ratio is constant over time (PH holds).
A significant p-value for a term indicates a PH violation for that variable.
Global test p-value is the omnibus test across all terms.
If the global test is significant, consider: stratification, time-varying
coefficients (tt() in coxph), or reporting with a caveat.

In [4]:
hideVars <- c("Constant", "distScot", "area", "uplands", "lowlands", "mean_slope")

cox_label_suffix_vars <- c("smHouse", "bigHouse", "friary",
                           "mg_fsnub", "mg_court",
                           "wet_1535", "wet_1536", "lLStax_pc", "lpopC")

cox_labels_by_spec <- lapply(cox_mon_specs, function(mon_vars) {
  c(mon_vars, cox_label_suffix_vars)
})

for (spec_name in names(cox_fits)) {
  sfx  <- cox_spec_suffix[[spec_name]]
  mods <- cox_fits[[spec_name]]

  cat(sprintf(
    "\n========== PH TESTS [%s] ==========\n\n", toupper(spec_name)))
  cat("--- Model 1 (monastic + weather) ---\n");           print(cox.zph(mods$m1))
  cat("\n--- Model 2 (+ taxation/population) ---\n");      print(cox.zph(mods$m2))
  cat("\n--- Model 3 (+ geographic controls) ---\n")
  zph3s <- cox.zph(mods$m3)
  print(zph3s)

  png(paste0("Output/Images/Graphs/cox3_schoenfeld", sfx, ".png"),
      width = 1200, height = 900, res = 120)
  par(mfrow = c(3, 4))
  plot(zph3s)
  dev.off()

  # Conley SEs (100 km Bartlett) for each progressive Cox model
  se_m1 <- conley_cox_se(mods$m1)
  se_m2 <- conley_cox_se(mods$m2)
  se_m3 <- conley_cox_se(mods$m3)

  stargazer(mods$m1, mods$m2, mods$m3,
    type = "latex",
    title = paste0("Risk of Rebellion — Cox Proportional Hazards [", spec_name, "]"),
    se = list(se_m1, se_m2, se_m3),
    omit = hideVars,
    align = TRUE,
    table.placement = "H",
    column.labels = c("Land", "Taxation and Population", "Geographic Controls"),
    add.lines = list(
      c("Population",           "N", "Y", "Y"),
      c("Geographic Controls",  "N", "N", "Y"),
      c("Conley SEs (100 km)",  "Y", "Y", "Y")
    ),
    covariate.labels = unlist(pretty_dict[cox_labels_by_spec[[spec_name]]]),
    omit.stat = c("wald", "lr", "logrank"),
    out = paste0("Output/Tables/survival", sfx, ".tex")
  )
}

# Back-compat: keep the split_arak zph objects under the old names
zph1 <- cox.zph(cox1)
zph2 <- cox.zph(cox2)
zph3 <- cox.zph(cox3)


========== PH TESTS [TOTAL_RAW] ==========

--- Model 1 (monastic + weather) ---
              chisq df      p
llandOwned 1.05e+00  1 0.3060
ltitheOutT 5.89e-04  1 0.9806
lalmsInTot 6.52e-01  1 0.4195
smHouse    1.39e+00  1 0.2390
bigHouse   3.68e+00  1 0.0550
friary     2.46e-01  1 0.6199
wet_1535   1.03e+01  1 0.0013
wet_1536   5.78e-01  1 0.4472
GLOBAL     1.69e+01  8 0.0313

--- Model 2 (+ taxation/population) ---
              chisq df      p
llandOwned 3.62e-01  1 0.5473
ltitheOutT 6.93e-02  1 0.7923
lalmsInTot 4.64e-01  1 0.4958
smHouse    1.92e+00  1 0.1656
bigHouse   5.47e+00  1 0.0193
friary     4.32e-01  1 0.5112
mg_fsnub   5.74e+00  1 0.0166
mg_court   2.32e+00  1 0.1278
wet_1535   9.93e+00  1 0.0016
wet_1536   3.84e-02  1 0.8447
lLStax_pc  5.90e-07  1 0.9994
lpopC      2.71e+00  1 0.0997
GLOBAL     2.14e+01 12 0.0450

--- Model 3 (+ geographic controls) ---
              chisq df      p
llandOwned 3.69e-01  1 0.5438
ltitheOutT 6.81e-02  1 0.7941
lalmsInTot 4.56e-01  1 0.4

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:22
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [total_raw]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{Lan

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:22
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [total_sk]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{Land

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:23
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [total_arak]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{La

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:24
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [split_raw]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{Lan

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:25
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [split_sk]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{Land

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:25
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [split_arak]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{La

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:26
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [ownOther_raw]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:27
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [ownOther_sk]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}{L

conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 
conleyreg (cox) failed: 'arg' should be one of "ols", "logit", "probit", "poisson" 

% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:12:28
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Risk of Rebellion — Cox Proportional Hazards [ownOther_arak]} 
  \label{} 
\begin{tabular}{@{\extracolsep{5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} & \multicolumn{1}{c}{"Surv(primary\_survival, primary) \textasciitilde"} \\ 
 & \multicolumn{1}{c}

## Section 2: Cox Model Coefficient Plots

In [5]:
# Shared blocks added to each progressive model's plot variable list
plot_middle  <- c("smHouse", "bigHouse", "friary")
plot_elite   <- c("mg_fsnub", "mg_court")
plot_weather <- c("wet_1535", "wet_1536")
plot_taxpop  <- c("lLStax_pc", "lpopC")

# Back-compat: per_km → split_arak variable sets for external reference
vars_cox1 <- c(cox_mon_specs$split_arak, plot_middle, plot_weather)
vars_cox2 <- c(cox_mon_specs$split_arak, plot_middle, plot_elite,
               plot_weather, plot_taxpop)
vars_cox3 <- vars_cox2

make_vars_trio <- function(mon_vars) {
  list(
    m1 = c(mon_vars, plot_middle, plot_weather),
    m2 = c(mon_vars, plot_middle, plot_elite, plot_weather, plot_taxpop),
    m3 = c(mon_vars, plot_middle, plot_elite, plot_weather, plot_taxpop)
  )
}
cox_plot_vars <- lapply(cox_mon_specs, make_vars_trio)

# Function to extract coefficients with 90% CI from coxph
extract_coefs_coxph <- function(model, var_name) {
  coef_summary <- summary(model)$coefficients
  coef_val <- coef_summary[var_name, "coef"]
  se_val   <- coef_summary[var_name, "se(coef)"]
  ci_lower <- coef_val - 1.645 * se_val  # 90% CI
  ci_upper <- coef_val + 1.645 * se_val
  p_val    <- coef_summary[var_name, "Pr(>|z|)"]
  data.frame(variable = var_name, coefficient = coef_val, se = se_val,
             ci_lower = ci_lower, ci_upper = ci_upper, p_value = p_val)
}

make_coef_df <- function(model, vars) {
  coefs <- bind_rows(lapply(vars, function(v) extract_coefs_coxph(model, v)))
  labels_vec <- unlist(pretty_dict)
  coefs$variable_label <- unname(labels_vec[coefs$variable])
  coefs$significant    <- coefs$p_value < 0.10
  coefs$order          <- match(coefs$variable, vars)
  coefs
}

make_cox_plot <- function(coef_df, x_label = "Coefficient (Log Hazard Ratio)") {
  lvl_order <- order(-coef_df$order)
  coef_df$variable_label <- factor(coef_df$variable_label,
                                    levels = coef_df$variable_label[lvl_order])
  ggplot(coef_df, aes(x = coefficient, y = variable_label)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "gray50") +
    geom_errorbar(aes(xmin = ci_lower, xmax = ci_upper),
                  width = 0.2, color = "gray30", orientation = "y") +
    geom_point(aes(color = significant), size = 3) +
    scale_color_manual(
      values = c("FALSE" = "gray60", "TRUE" = "#0072B2"),
      labels = c("FALSE" = "Not Significant", "TRUE" = "p < 0.10")
    ) +
    labs(x = x_label, y = "", color = "Significance") +
    theme_minimal() +
    theme(
      axis.text.x  = element_text(size = 16),
      axis.text.y  = element_text(size = 16),
      axis.title.x = element_text(size = 16),
      legend.text  = element_text(size = 15),
      legend.title = element_text(size = 15),
      legend.position = "bottom"
    )
}

for (spec_name in names(cox_fits)) {
  sfx     <- cox_spec_suffix[[spec_name]]
  mods    <- cox_fits[[spec_name]]
  vbundle <- cox_plot_vars[[spec_name]]

  for (k in c("m1", "m2", "m3")) {
    coefs <- make_coef_df(mods[[k]], vbundle[[k]])
    idx   <- substr(k, 2, 2)  # "1"/"2"/"3"
    ggsave(
      paste0("Output/Images/Graphs/cox", idx, "_coefficients", sfx, ".png"),
      plot = make_cox_plot(coefs), width = 10, height = 6, dpi = 300
    )
  }
}

cat("\nSurvival analysis outputs created successfully!\n")
cat("Tables:  Output/Tables/survival_{total,split,ownOther}_{raw,sk,arak}.tex\n")
cat("Plots:   Output/Images/Graphs/cox{1,2,3}_coefficients_{...}.png\n")



Survival analysis outputs created successfully!


Tables:  Output/Tables/survival_{total,split,ownOther}_{raw,sk,arak}.tex


Plots:   Output/Images/Graphs/cox{1,2,3}_coefficients_{...}.png
